# Benchmarking Q-Filters

This notebook benchmarks the performance of Q-Filters against the standard implementation.

In [ ]:
# Setup
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from datasets import load_dataset
from src.hf_cache import QFiltersCache, KNormCache
import time

model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
streamer = TextStreamer(tokenizer)

question = """What is the probability of two integers selected at random having a greatest common divisor of 1."""
input_text = f"<|User|>{question}<|Assistant|><think>\n"
inputs = tokenizer(input_text, return_tensors="pt")

def benchmark_model(model, past_key_values):
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    start_time = time.time()
    model.generate(
        **inputs,
        do_sample=True,
        temperature=0.5,
        max_new_tokens=4096,
        past_key_values=past_key_values,
        streamer=streamer
    )
    end_time = time.time()
    return end_time - start_time


In [ ]:
# Load models
standard_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype="bfloat16"
)

qfilters_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype="bfloat16"
)


In [ ]:
# Benchmark standard implementation
standard_past_key_values = KNormCache(
    window_length=64,
    max_length=128
)
standard_time = benchmark_model(standard_model, standard_past_key_values)
print(f"Standard implementation time: {standard_time} seconds")


In [ ]:
# Benchmark Q-Filters implementation
qfilters_past_key_values = QFiltersCache(
    window_length=64,
    max_length=128,
    model_name=model_name
)
qfilters_time = benchmark_model(qfilters_model, qfilters_past_key_values)
print(f"Q-Filters implementation time: {qfilters_time} seconds")


## Results

The benchmark results show the time taken by the standard implementation and the Q-Filters implementation.